# 📈 Módulo 04 · Matplotlib

Visualización con criterio: qué gráfico responde cada pregunta, cómo hacerlo legible y cómo interpretarlo sin engañar.

Versión web con conceptos, reconocimiento, razonamiento y proyecto: [https://rosalesluciano.github.io/datalab/](https://rosalesluciano.github.io/datalab/#/m/m04)

## Cómo trabajar este notebook
1. Ejecutá la celda ⚙️ (carga datos y verificador).
2. **Antes de escribir código**, escribí en un comentario cómo lo vas a resolver.
3. Implementá y guardá tu respuesta en `resultado` (o lo que pida el ejercicio).
4. `comprobar()` verifica. Si te trabás: `ver_pista("id", 1)`, después 2 y 3.
5. `ver_solucion("id")` es el último recurso: muestra idea → pseudocódigo → código.

## Conceptos del módulo
- **Elegir el gráfico según la pregunta**: Elegir la visualización correcta: cada tipo de gráfico responde un tipo de pregunta.
- **Gráfico de líneas**: Mostrar cómo cambia una variable a lo largo de algo ordenado, típicamente el tiempo.
- **Gráfico de barras**: Comparar una cantidad entre categorías.
- **Histograma**: Ver cómo se **distribuye** una variable numérica: dónde se concentran los valores, si hay asimetría o valores extremos.
- **Gráfico de dispersión**: Estudiar la **relación** entre dos variables numéricas: si crecen juntas, si hay grupos u outliers.
- **Anatomía de un gráfico legible**: Que el gráfico se entienda solo: qué se muestra, en qué unidades y qué es cada serie.
- **Varios gráficos: figura y ejes**: Combinar varios gráficos en una figura (un mini dashboard) para comparar vistas de los mismos datos.
- **Graficar desde pandas**: Hacer gráficos rápidos directamente desde un DataFrame o Series (pandas usa matplotlib por debajo).
- **Interpretar (y no engañar)**: Sacar conclusiones correctas de un gráfico y detectar gráficos engañosos.

In [ ]:
#@title ⚙️ 1) Ejecutá esta celda primero (datos y verificador) { display-mode: "form" }
"""PandasQuest - motor de corrección.

Se usa igual en el navegador (Pyodide), en los notebooks de Colab y en los tests.
Compara el `resultado` del alumno contra el de la solución oficial con tolerancia
a diferencias irrelevantes (tipos int/float, nombres de índice, dtype string, etc.).
"""
import math

import numpy as np
import pandas as pd


def _norm_s(s):
    dt = s.dtype
    if isinstance(dt, pd.CategoricalDtype) or pd.api.types.is_string_dtype(dt):
        s = s.astype(object)
    elif pd.api.types.is_extension_array_dtype(dt):
        if pd.api.types.is_numeric_dtype(dt) and not pd.api.types.is_bool_dtype(dt):
            s = s.astype("float64")
        else:
            s = s.astype(object)
    if s.dtype == object:
        s = s.where(s.notna(), None)
    return s


def _norm_index(idx):
    if isinstance(idx, pd.MultiIndex):
        return idx
    if isinstance(idx, pd.CategoricalIndex) or pd.api.types.is_string_dtype(idx.dtype):
        return pd.Index(list(idx), dtype=object)
    return idx


def _norm(o):
    if isinstance(o, pd.Series):
        o = _norm_s(o.copy())
    else:
        cols = list(o.columns)
        o = pd.DataFrame({i: _norm_s(o.iloc[:, i]) for i in range(o.shape[1])}, index=o.index)
        o.columns = _norm_index(pd.Index(cols))
    o.index = _norm_index(o.index)
    return o


def _sort(o, ignore_index):
    if ignore_index:
        o = o.reset_index(drop=True)
        try:
            if isinstance(o, pd.Series):
                o = o.sort_values(kind="mergesort")
            else:
                o = o.sort_values(by=list(o.columns), kind="mergesort")
        except TypeError:
            key = o.astype(str) if isinstance(o, pd.Series) else o.astype(str).agg("|".join, axis=1)
            o = o.iloc[np.argsort(key.to_numpy(), kind="mergesort")]
        return o.reset_index(drop=True)
    try:
        return o.sort_index()
    except TypeError:
        return o


def _tipo(x):
    nombres = {pd.DataFrame: "un DataFrame", pd.Series: "una Series", list: "una lista", dict: "un diccionario",
               tuple: "una tupla", set: "un set", str: "un texto (str)", int: "un entero (int)",
               float: "un decimal (float)", bool: "un booleano (bool)", type(None): "None (nada)",
               np.ndarray: "un array de NumPy"}
    return nombres.get(type(x), f"un {type(x).__name__}")


def _scalar_eq(u, e):
    if isinstance(e, (np.generic,)):
        e = e.item()
    if isinstance(u, (np.generic,)):
        u = u.item()
    if isinstance(e, float) or isinstance(u, float):
        try:
            u, e = float(u), float(e)
        except (TypeError, ValueError):
            return False
        if math.isnan(e):
            return math.isnan(u)
        return math.isclose(u, e, rel_tol=1e-6, abs_tol=1e-6)
    if isinstance(e, (list, tuple)) and isinstance(u, (list, tuple, np.ndarray, pd.Index)):
        u = list(u)
        return len(u) == len(e) and all(_scalar_eq(a, b) for a, b in zip(u, e))
    if isinstance(e, dict):
        return (isinstance(u, dict) and set(map(str, u)) == set(map(str, e))
                and all(_scalar_eq(u[k] if k in u else u[str(k)], v) for k, v in e.items()))
    if isinstance(e, (set, frozenset)):
        return isinstance(u, (set, frozenset)) and u == e
    try:
        return bool(u == e)
    except Exception:
        return False


def _array_compare(u, e):
    if isinstance(u, (pd.DataFrame, pd.Series)):
        u = u.to_numpy()
    try:
        ua = np.asarray(u)
    except Exception:
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if not isinstance(u, np.ndarray) and not isinstance(u, (list, tuple)):
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if ua.shape != e.shape:
        return False, f"Forma distinta: tu array tiene shape {ua.shape} y se esperaba {e.shape}."
    num = np.issubdtype(e.dtype, np.number) or e.dtype == bool
    try:
        if num and (np.issubdtype(ua.dtype, np.number) or ua.dtype == bool):
            ok = np.allclose(ua.astype(float), e.astype(float), rtol=1e-6, atol=1e-6, equal_nan=True)
        else:
            ok = ua.tolist() == e.tolist()
    except Exception:
        ok = False
    return (True, "") if ok else (False, "La forma es correcta pero los valores no coinciden. Revisa el cálculo.")


def pq_compare(u, e, ordered=True, ignore_index=False):
    """Devuelve (ok, mensaje)."""
    if isinstance(e, np.ndarray):
        return _array_compare(u, e)
    if isinstance(e, (pd.DataFrame, pd.Series)):
        if not isinstance(u, type(e)):
            extra = ""
            if isinstance(e, pd.Series) and isinstance(u, pd.DataFrame):
                extra = " Pista: con una sola columna usa df['col'] (corchetes simples)."
            if isinstance(e, pd.DataFrame) and isinstance(u, pd.Series):
                extra = " Pista: usa doble corchete df[['col']] o .reset_index() / .to_frame()."
            return False, f"Tu resultado es {_tipo(u)}, pero se esperaba {_tipo(e)}.{extra}"
        if isinstance(e, pd.DataFrame):
            uc, ec = [str(c) for c in u.columns], [str(c) for c in e.columns]
            if uc != ec:
                if sorted(uc) == sorted(ec):
                    return False, f"Tienes las columnas correctas pero en otro orden. Se esperaba: {ec}"
                faltan = [c for c in ec if c not in uc]
                sobran = [c for c in uc if c not in ec]
                msg = "Las columnas no coinciden."
                if faltan:
                    msg += f" Faltan: {faltan}."
                if sobran:
                    msg += f" Sobran: {sobran}."
                return False, msg
        if u.shape != e.shape:
            return False, f"Forma distinta: tu resultado tiene {u.shape} y se esperaba {e.shape} (filas, columnas)."
        u2, e2 = _norm(u), _norm(e)
        if not ordered:
            u2, e2 = _sort(u2, ignore_index), _sort(e2, ignore_index)
        elif ignore_index:
            u2, e2 = u2.reset_index(drop=True), e2.reset_index(drop=True)
        kw = dict(check_dtype=False, check_names=False, check_index_type=False,
                  check_exact=False, rtol=1e-6, atol=1e-6)
        try:
            if isinstance(e2, pd.DataFrame):
                pd.testing.assert_frame_equal(u2, e2, check_column_type=False, check_freq=False, **kw)
            else:
                pd.testing.assert_series_equal(u2, e2, check_freq=False, **kw)
        except AssertionError as err:
            txt = str(err)
            if "index" in txt.lower() and "values are different" in txt.lower() and "iloc" not in txt:
                return False, "Los valores parecen correctos pero el índice no coincide. ¿Hace falta ordenar o usar reset_index()?"
            if ordered and not ignore_index:
                try:
                    pd.testing.assert_frame_equal(
                        u2.to_frame() if isinstance(u2, pd.Series) else u2,
                        e2.to_frame() if isinstance(e2, pd.Series) else e2,
                        check_like=True, **kw)
                except AssertionError:
                    pass
                else:
                    return False, "Mismos datos, pero en otro orden. Revisa cómo ordenas las filas."
            return False, "Los valores no coinciden con lo esperado. Revisa la lógica y vuelve a intentarlo."
        return True, ""
    if isinstance(u, (pd.DataFrame, pd.Series)):
        return False, f"Tu resultado es {_tipo(u)}, pero se esperaba un valor simple ({type(e).__name__}). ¿Te falta .iloc[0], .sum() o similar?"
    if _scalar_eq(u, e):
        return True, ""
    if u is None:
        return False, ("Tu resultado es None. ¿Te falta un `return` en la función, o guardaste lo que devuelve "
                       "un método que modifica en el lugar (como `.sort()` o `.append()`)?")
    contenedores = (list, tuple, dict, set, str)
    if isinstance(e, contenedores) and not isinstance(u, type(e)) and not (
            isinstance(e, (list, tuple)) and isinstance(u, (list, tuple))):
        return False, f"Tu resultado es {_tipo(u)}, pero se esperaba {_tipo(e)}."
    if isinstance(e, (list, tuple)) and isinstance(u, (list, tuple)) and len(u) != len(e):
        return False, f"Tu resultado tiene {len(u)} elementos y se esperaban {len(e)}."
    if isinstance(e, dict) and isinstance(u, dict) and set(map(str, u)) != set(map(str, e)):
        faltan = [k for k in e if str(k) not in set(map(str, u))]
        sobran = [k for k in u if str(k) not in set(map(str, e))]
        msg = "Las claves del diccionario no coinciden."
        if faltan:
            msg += f" Faltan: {faltan}."
        if sobran:
            msg += f" Sobran: {sobran}."
        return False, msg
    txt = repr(u)
    return False, f"Tu resultado es {txt[:300] + ('…' if len(txt) > 300 else '')}, pero no es el valor esperado."


# ---------------------------------------------------------------- gráficos
def _plt():
    import matplotlib.pyplot as plt
    return plt


def _fig():
    plt = _plt()
    return plt.gcf() if plt.get_fignums() else None


def _ax(i=0):
    f = _fig()
    if f is None or len(f.axes) <= i:
        raise LookupError("no hay ningún gráfico dibujado")
    return f.axes[i]


def _legend(ax=None):
    ax = ax or _ax()
    lg = ax.get_legend()
    return [t.get_text() for t in lg.get_texts()] if lg else []


def _heights(ax=None):
    return [round(float(p.get_height()), 6) for p in (ax or _ax()).patches]


def _widths(ax=None):
    return [round(float(p.get_width()), 6) for p in (ax or _ax()).patches]


def _xticks(ax=None):
    return [t.get_text() for t in (ax or _ax()).get_xticklabels()]


def _hex(color):
    import matplotlib.colors as mcolors
    return mcolors.to_hex(color)


_HELPERS = {"_fig": _fig, "_ax": _ax, "_legend": _legend, "_heights": _heights,
            "_widths": _widths, "_xticks": _xticks, "_hex": _hex}


def pq_prepare():
    """Deja matplotlib limpio antes de ejecutar código (si está cargado)."""
    import sys
    if "matplotlib.pyplot" in sys.modules:
        plt = sys.modules["matplotlib.pyplot"]
        plt.close("all")
        plt.show = lambda *a, **k: None


def pq_check(ns, get_expected, opts):
    """Evalúa el ejercicio.

    ns = namespace del alumno. get_expected = función que devuelve el namespace de la
    solución (solo se ejecuta si hace falta, para no pisar los gráficos del alumno).
    """
    custom = opts.get("custom")
    if custom:
        if opts.get("plot") and _fig() is None:
            return False, "Todavía no hay ningún gráfico. ¡Dibuja algo! 🎨"
        env = dict(_HELPERS)
        env.update(ns)
        try:
            ok = bool(eval(custom, env))
        except Exception as err:
            return False, f"{opts.get('custom_msg', 'Todavía no es correcto.')} ({type(err).__name__}: {err})"
        return ok, "" if ok else opts.get("custom_msg", "Todavía no es correcto. ¡Sigue intentándolo!")
    if "resultado" not in ns:
        return False, "Guarda tu respuesta en la variable `resultado`."
    exp_ns = get_expected() if callable(get_expected) else get_expected
    return pq_compare(ns["resultado"], exp_ns["resultado"],
                      ordered=opts.get("ordered", True),
                      ignore_index=opts.get("ignore_index", False))


_ERR_TIPS = [
    ("truth value of a Series is ambiguous", "Para combinar condiciones usá & (y), | (o) y poné cada condición entre paréntesis."),
    ("can't multiply sequence", "Estás multiplicando un texto (str). ¿El número llegó como texto? Convertilo con int() o float()."),
    ("can only concatenate str", "Estás sumando texto con otra cosa. Convertí con str() para unir textos, o con float() para hacer cuentas."),
    ("unsupported operand type", "Los dos lados de la operación tienen tipos incompatibles (por ejemplo, int + str o None + float). Revisá type() de cada valor."),
    ("could not convert string to float", "Hay un texto que no representa un número (por ejemplo 'n/d' o ''). Filtralo o capturá el ValueError con try/except."),
    ("invalid literal for int", "Hay un texto que no representa un entero. Revisá el valor o usá try/except ValueError."),
    ("index out of range", "Pediste una posición que no existe. Recordá: las posiciones van de 0 a len()-1, y -1 es el último."),
    ("values to unpack", "La cantidad de variables a la izquierda no coincide con la cantidad de valores que llegan."),
    ("ZeroDivisionError", "Dividiste por cero. ¿La lista o el divisor pueden estar vacíos? Verificalo antes de dividir."),
    ("'NoneType' object", "Un valor es None. Suele pasar con una función sin return o con métodos como .sort() que devuelven None."),
    ("KeyError", "Esa clave o columna no existe. Revisá mayúsculas, tildes y espacios; con dicts podés usar .get()."),
    ("NameError", "Usás una variable que no existe (todavía). ¿Error de tipeo o la definiste más abajo?"),
    ("IndentationError", "Revisá la sangría: el contenido de if, for, def, etc. va con 4 espacios."),
    ("SyntaxError", "Error de sintaxis: revisá paréntesis, comillas, dos puntos (:) al final de if/for/def y que cada línea esté completa."),
    ("AttributeError", "Ese método o atributo no existe para este tipo de objeto. Revisá el nombre y qué tipo tiene la variable."),
    ("MergeError", "Revisá las columnas que usás para unir (on / left_on / right_on)."),
    ("TypeError", "Algún tipo de dato no encaja con la operación. Revisá type() de las variables involucradas."),
    ("ValueError", "El tipo es correcto pero el valor no es válido para esta operación."),
]


def pq_error_tip(msg):
    for key, tip in _ERR_TIPS:
        if key in msg:
            return tip
    return ""


_SETUP = 'import pandas as pd\nimport numpy as np\n\nclientes = pd.DataFrame({\n    "cliente_id": range(1, 13),\n    "nombre": ["Ana", "Bruno", "Carla", "Diego", "Elena", "Facundo",\n               "Gabriela", "Hugo", "Inés", "Julián", "Karina", "Lucas"],\n    "ciudad": ["Madrid", "Buenos Aires", "CDMX", "Bogotá", "Madrid", "Lima",\n               "Buenos Aires", "CDMX", "Santiago", "Bogotá", "Madrid", "Lima"],\n    "pais": ["España", "Argentina", "México", "Colombia", "España", "Perú",\n             "Argentina", "México", "Chile", "Colombia", "España", "Perú"],\n    "edad": [34, 28, 45, 23, 51, 39, 30, 27, 42, 36, 25, 48],\n    "segmento": ["Premium", "Básico", "Premium", "Básico", "Premium", "Estándar",\n                 "Estándar", "Básico", "Premium", "Estándar", "Básico", "Estándar"],\n    "email": ["ana@mail.com", None, "carla@mail.com", "diego@mail.com", None, "facu@mail.com",\n              "gabi@mail.com", None, "ines@mail.com", "julian@mail.com", "kari@mail.com", "lucas@mail.com"],\n    "fecha_registro": pd.to_datetime([\n        "2023-01-15", "2023-02-03", "2023-02-20", "2023-03-11", "2023-04-02", "2023-05-19",\n        "2023-06-07", "2023-07-23", "2023-08-30", "2023-09-14", "2023-10-05", "2023-11-28"]),\n})\n\nproductos = pd.DataFrame({\n    "producto_id": range(1, 11),\n    "producto": ["Laptop", "Mouse", "Teclado", "Monitor", "Auriculares",\n                 "Webcam", "Silla gamer", "Escritorio", "Tablet", "Micrófono"],\n    "categoria": ["Computación", "Accesorios", "Accesorios", "Computación", "Audio",\n                  "Accesorios", "Muebles", "Muebles", "Computación", "Audio"],\n    "precio": [1200.0, 25.5, 45.0, 300.0, 80.0, 60.0, 250.0, 180.0, 450.0, 95.0],\n    "stock": [5, 150, 80, 20, 60, 0, 12, 7, 15, 0],\n})\n\n_estados = ["entregado", "enviado", "entregado", "cancelado", "entregado", "pendiente"]\npedidos = pd.DataFrame({\n    "pedido_id": range(1001, 1031),\n    "cliente_id": [(i * 7) % 11 + 1 for i in range(29)] + [99],\n    "producto_id": [(i * 3) % 10 + 1 for i in range(30)],\n    "cantidad": [(i % 4) + 1 for i in range(30)],\n    "fecha": pd.date_range("2024-01-03", periods=30, freq="5D"),\n    "estado": [_estados[i % 6] for i in range(30)],\n})\n\nempleados = pd.DataFrame({\n    "emp_id": range(1, 11),\n    "nombre": ["Marta", "Pablo", "Sofía", "Tomás", "Valeria",\n               "Andrés", "Lucía", "Martín", "Paula", "Ramiro"],\n    "depto": ["Dirección", "Ventas", "Ventas", "Ventas", "IT",\n              "IT", "IT", "Marketing", "Marketing", "Ventas"],\n    "salario": [9000, 4200, 3900, 4000, 5200, 4800, 6100, 3700, 4100, 3500],\n    "jefe_id": pd.array([None, 1, 2, 2, 1, 5, 5, 1, 8, 2], dtype="Int64"),\n    "fecha_ingreso": pd.to_datetime([\n        "2015-03-01", "2018-06-15", "2019-01-10", "2020-09-01", "2017-11-20",\n        "2021-02-01", "2016-08-08", "2022-04-18", "2019-07-07", "2023-01-09"]),\n})\n\n_i = np.arange(90)\n_f = pd.date_range("2024-01-01", periods=90, freq="D")\n_v = 1000 + 8 * _i + 250 * (_f.dayofweek >= 5) + ((_i * 37) % 11) * 20\n_v = np.where(np.isin(_i, [20, 55, 77]), _v + 1500, _v)\nventas_diarias = pd.DataFrame({\n    "fecha": _f,\n    "ventas": _v.astype(float),\n    "visitas": (300 + (_i * 13) % 97 + 2 * _i).astype(int),\n})\n\nsucio = pd.DataFrame({\n    "id": [1, 2, 2, 3, 4, 5, 6, 7],\n    "nombre": ["  ana ", "BRUNO", "BRUNO", "carla", None, "Diego  ", "elena", "facundo"],\n    "edad": ["34", "28", "28", "n/d", "23", "51", None, "39"],\n    "ciudad": ["madrid", "Buenos Aires", "Buenos Aires", " santiago", "bogotá", "Madrid ", "madrid", "LIMA"],\n    "monto": ["$1,200.50", "$300", "$300", "$45.00", None, "$80.5", "$99", "$1,000"],\n})\ndel _i, _f, _v, _estados\n\nimport matplotlib.pyplot as plt\n\nmeses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun"]\ningresos = [12.5, 14.1, 13.8, 16.2, 18.9, 21.3]   # miles\ngastos = [10.2, 11.0, 11.5, 12.1, 13.0, 13.4]     # miles\n'
_SOL_B64 = 'eyJwbHQwMSI6IHsic29sdXRpb24iOiAicGx0LnBsb3QobWVzZXMsIGluZ3Jlc29zKSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAibGVuKF9heCgpLmxpbmVzKSA9PSAxIGFuZCBsaXN0KF9heCgpLmxpbmVzWzBdLmdldF95ZGF0YSgpKSA9PSBpbmdyZXNvcyIsICJjdXN0b21fbXNnIjogIkRlYmUgaGFiZXIgdW5hIGzDrW5lYSBjb24gbG9zIHZhbG9yZXMgZGUgYGluZ3Jlc29zYCBlbiBlbCBlamUgWS4ifSwgImhpbnRzIjogWyJFcyB1bmEgZXZvbHVjacOzbiBtZXMgYSBtZXM6IMK/cXXDqSBncsOhZmljbz8iLCAiRW4gWCB2YW4gbG9zIG1lc2VzIHkgZW4gWSBsb3MgaW5ncmVzb3MuIiwgInBsdC5wbG90KHgsIHkpIl0sICJleHBsYWluIjogImBwbHQucGxvdCh4LCB5KWAgZGlidWphIHVuYSBsw61uZWEgdW5pZW5kbyBsb3MgcHVudG9zIChtZXMsIGluZ3Jlc28pIGVuIG9yZGVuLiIsICJwc2V1ZG8iOiAibMOtbmVhKG1lc2VzLCBpbmdyZXNvcykifSwgInBsdDAyIjogeyJzb2x1dGlvbiI6ICJwbHQucGxvdChtZXNlcywgaW5ncmVzb3MpXG5wbHQudGl0bGUoJ0luZ3Jlc29zIHBvciBtZXMnKVxucGx0LnhsYWJlbCgnTWVzJylcbnBsdC55bGFiZWwoJ0luZ3Jlc29zIChtaWxlcyknKSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAibGVuKF9heCgpLmxpbmVzKSA9PSAxIGFuZCBfYXgoKS5nZXRfdGl0bGUoKSA9PSAnSW5ncmVzb3MgcG9yIG1lcycgYW5kIF9heCgpLmdldF94bGFiZWwoKSA9PSAnTWVzJyBhbmQgX2F4KCkuZ2V0X3lsYWJlbCgpID09ICdJbmdyZXNvcyAobWlsZXMpJyIsICJjdXN0b21fbXNnIjogIlJldmlzYSBxdWUgZWwgdMOtdHVsbyB5IGxhcyBldGlxdWV0YXMgZXN0w6luIGVzY3JpdG9zIGV4YWN0YW1lbnRlIGNvbW8gc2UgcGlkZS4ifSwgImhpbnRzIjogWyJFbCBncsOhZmljbyB0aWVuZSBxdWUgZW50ZW5kZXJzZSBzaW4gZXhwbGljYWNpw7NuLiIsICJGYWx0YW4gdMOtdHVsbyB5IG5vbWJyZXMgZGUgZWplcy4iLCAicGx0LnRpdGxlLCBwbHQueGxhYmVsLCBwbHQueWxhYmVsIChjb24gZWwgdGV4dG8gZXhhY3RvKS4iXSwgImV4cGxhaW4iOiAiVMOtdHVsbyB5IGV0aXF1ZXRhcyBjb252aWVydGVuIHVuYSBsw61uZWEgZW4gdW4gZ3LDoWZpY28gaW50ZXJwcmV0YWJsZTogcXXDqSBzZSBtdWVzdHJhIHkgZW4gcXXDqSB1bmlkYWRlcy4iLCAicHNldWRvIjogImzDrW5lYShtZXNlcywgaW5ncmVzb3MpXG50w610dWxvICdJbmdyZXNvcyBwb3IgbWVzJzsgZWplIFggJ01lcyc7IGVqZSBZICdJbmdyZXNvcyAobWlsZXMpJyJ9LCAicGx0MDMiOiB7InNvbHV0aW9uIjogInBsdC5wbG90KG1lc2VzLCBpbmdyZXNvcywgbGFiZWw9J0luZ3Jlc29zJylcbnBsdC5wbG90KG1lc2VzLCBnYXN0b3MsIGxhYmVsPSdHYXN0b3MnKVxucGx0LmxlZ2VuZCgpIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICJsZW4oX2F4KCkubGluZXMpID09IDIgYW5kIHNvcnRlZChfbGVnZW5kKCkpID09IFsnR2FzdG9zJywgJ0luZ3Jlc29zJ10iLCAiY3VzdG9tX21zZyI6ICJEZWJlbiB2ZXJzZSAyIGzDrW5lYXMgeSB1bmEgbGV5ZW5kYSBjb24gJ0luZ3Jlc29zJyB5ICdHYXN0b3MnLiJ9LCAiaGludHMiOiBbIlNvbiBkb3Mgc2VyaWVzIHNvYnJlIGVsIG1pc21vIGVqZSBkZSB0aWVtcG8uIiwgIkRvcyBsbGFtYWRhcyBhIHBsb3QgZW4gZWwgbWlzbW8gZ3LDoWZpY287IGNhZGEgdW5hIGNvbiBzdSBsYWJlbC4iLCAicGx0LmxlZ2VuZCgpIG11ZXN0cmEgbG9zIGxhYmVscy4iXSwgImV4cGxhaW4iOiAiQ2FkYSBgcGxvdGAgYWdyZWdhIHVuYSBsw61uZWE7IGBsYWJlbD1gIGxlIGRhIG5vbWJyZSB5IGBsZWdlbmQoKWAgbXVlc3RyYSBsYSBsZXllbmRhIHBhcmEgZGlzdGluZ3Vpcmxhcy4iLCAicHNldWRvIjogImzDrW5lYShtZXNlcywgaW5ncmVzb3MsIGxhYmVsPSdJbmdyZXNvcycpXG5sw61uZWEobWVzZXMsIGdhc3RvcywgbGFiZWw9J0dhc3RvcycpXG5sZXllbmRhIn0sICJwbHQtZTAxIjogeyJzb2x1dGlvbiI6ICJwbHQucGxvdChtZXNlcywgaW5ncmVzb3MsIGxhYmVsPSdJbmdyZXNvcycpXG5wbHQucGxvdChtZXNlcywgZ2FzdG9zLCBsYWJlbD0nR2FzdG9zJylcbnBsdC5sZWdlbmQoKSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAibGVuKF9heCgpLmxpbmVzKSA9PSAyIGFuZCBzb3J0ZWQoX2xlZ2VuZCgpKSA9PSBbJ0dhc3RvcycsICdJbmdyZXNvcyddIiwgImN1c3RvbV9tc2ciOiAiRGViZW4gdmVyc2UgMiBsw61uZWFzIHkgdW5hIGxleWVuZGEgY29uICdJbmdyZXNvcycgeSAnR2FzdG9zJy4ifSwgImhpbnRzIjogWyJNaXLDoSBsYSBzYWxpZGE6IMK/aGF5IGFsZ8O6biBhdmlzbz8iLCAiwr9EZSBkw7NuZGUgc2FjYSBsYSBsZXllbmRhIGxvcyBub21icmVzPyIsICJBZ3JlZ8OhIHVuIHBhcsOhbWV0cm8gYSBjYWRhIHBsb3QuIl0sICJleHBsYWluIjogImBwbHQubGVnZW5kKClgIG11ZXN0cmEgZWwgYGxhYmVsYCBkZSBjYWRhIHNlcmllLiBTaSBuaW5ndW5hIHNlcmllIHRpZW5lIGxhYmVsLCBubyBoYXkgbmFkYSBxdWUgbW9zdHJhciAobWF0cGxvdGxpYiBhdmlzYSAnTm8gYXJ0aXN0cyB3aXRoIGxhYmVscyBmb3VuZCcpLiIsICJwc2V1ZG8iOiAibMOtbmVhKC4uLiwgbGFiZWw9J0luZ3Jlc29zJyk7IGzDrW5lYSguLi4sIGxhYmVsPSdHYXN0b3MnKTsgbGV5ZW5kYSJ9LCAicGx0LWUwMiI6IHsic29sdXRpb24iOiAicGx0LnBsb3QobWVzZXMsIGluZ3Jlc29zKSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAibGVuKF9heCgpLmxpbmVzKSA9PSAxIGFuZCBsaXN0KF9heCgpLmxpbmVzWzBdLmdldF95ZGF0YSgpKSA9PSBpbmdyZXNvcyIsICJjdXN0b21fbXNnIjogIkRlYmUgaGFiZXIgdW5hIGzDrW5lYSBjb24gbG9zIG1lc2VzIGVuIFggeSBsb3MgaW5ncmVzb3MgZW4gWS4ifSwgImhpbnRzIjogWyJNaXLDoSBlbCBncsOhZmljbzogwr9kw7NuZGUgcXVlZGFyb24gbG9zIG1lc2VzPyIsICLCv1F1w6kgYXJndW1lbnRvIHZhIGFsIGVqZSBYPyIsICJJbnZlcnTDrSBlbCBvcmRlbi4iXSwgImV4cGxhaW4iOiAiYHBsdC5wbG90KHgsIHkpYDogZWwgcHJpbWVyIGFyZ3VtZW50byB2YSBhbCBlamUgaG9yaXpvbnRhbC4gUG9yIGNvbnZlbmNpw7NuLCBlbCB0aWVtcG8gKG1lc2VzKSB2YSBlbiBYIHkgbGEgdmFyaWFibGUgbWVkaWRhIGVuIFkuIiwgInBzZXVkbyI6ICJsw61uZWEobWVzZXMsIGluZ3Jlc29zKSJ9LCAicGx0MDQiOiB7InNvbHV0aW9uIjogInBsdC5wbG90KG1lc2VzLCBpbmdyZXNvcywgY29sb3I9J2dyZWVuJywgbWFya2VyPSdvJywgbGluZXN0eWxlPSctLScpIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICJsZW4oX2F4KCkubGluZXMpID09IDEgYW5kIF9oZXgoX2F4KCkubGluZXNbMF0uZ2V0X2NvbG9yKCkpID09ICcjMDA4MDAwJyBhbmQgX2F4KCkubGluZXNbMF0uZ2V0X21hcmtlcigpID09ICdvJyBhbmQgX2F4KCkubGluZXNbMF0uZ2V0X2xpbmVzdHlsZSgpID09ICctLSciLCAiY3VzdG9tX21zZyI6ICJMYSBsw61uZWEgZGViZSBzZXIgdmVyZGUgKCdncmVlbicpLCBjb24gbWFyY2Fkb3IgJ28nIHkgZXN0aWxvICctLScuIn0sICJoaW50cyI6IFsiU29uIHRyZXMgb3BjaW9uZXMgZGUgZXN0aWxvIGRlIGxhIG1pc21hIGzDrW5lYS4iLCAicGxvdCBhY2VwdGEgcGFyw6FtZXRyb3MgZGUgY29sb3IsIG1hcmNhZG9yIHkgZXN0aWxvIGRlIGzDrW5lYS4iLCAiY29sb3I9J2dyZWVuJywgbWFya2VyPSdvJywgbGluZXN0eWxlPSctLScuIl0sICJleHBsYWluIjogIkxvcyBwYXLDoW1ldHJvcyBkZSBlc3RpbG8gKGNvbG9yLCBtYXJrZXIsIGxpbmVzdHlsZSkgY2FtYmlhbiBjw7NtbyBzZSB2ZSBsYSBsw61uZWEgc2luIGNhbWJpYXIgbG9zIGRhdG9zLiIsICJwc2V1ZG8iOiAibMOtbmVhKG1lc2VzLCBpbmdyZXNvcywgdmVyZGUsIG1hcmNhZG9yICdvJywgZ3Vpb25lcyAnLS0nKSJ9LCAicGx0MDUiOiB7InNvbHV0aW9uIjogInBsdC5iYXIocHJvZHVjdG9zWydwcm9kdWN0byddLCBwcm9kdWN0b3NbJ3N0b2NrJ10pXG5wbHQueHRpY2tzKHJvdGF0aW9uPTQ1KSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAibGVuKF9heCgpLnBhdGNoZXMpID09IDEwIGFuZCBfaGVpZ2h0cygpID09IHByb2R1Y3Rvc1snc3RvY2snXS50b2xpc3QoKSIsICJjdXN0b21fbXNnIjogIkRlYmUgaGFiZXIgMTAgYmFycmFzIGNvbiBlbCBzdG9jayBkZSBjYWRhIHByb2R1Y3RvLCBlbiBlbCBvcmRlbiBkZSBsYSB0YWJsYS4ifSwgImhpbnRzIjogWyJDb21wYXLDoXMgdW5hIGNhbnRpZGFkIGVudHJlIGNhdGVnb3LDrWFzIChwcm9kdWN0b3MpLiIsICJCYXJyYXM6IGNhdGVnb3LDrWFzIGVuIFgsIHZhbG9yZXMgZW4gWS4iLCAicGx0LmJhcihjYXRlZ29yw61hcywgdmFsb3Jlcyk7IHJvdMOhIGxhcyBldGlxdWV0YXMgc2kgc2Ugc3VwZXJwb25lbi4iXSwgImV4cGxhaW4iOiAiYHBsdC5iYXJgIGRpYnVqYSB1bmEgYmFycmEgcG9yIHByb2R1Y3RvIGNvbiBhbHR1cmEgaWd1YWwgYSBzdSBzdG9jay4iLCAicHNldWRvIjogImJhcnJhcyhwcm9kdWN0b3NbJ3Byb2R1Y3RvJ10sIHByb2R1Y3Rvc1snc3RvY2snXSkifSwgInBsdDA2IjogeyJzb2x1dGlvbiI6ICJ0b3AgPSBwcm9kdWN0b3Mubmxhcmdlc3QoNSwgJ3ByZWNpbycpXG5wbHQuYmFyaCh0b3BbJ3Byb2R1Y3RvJ10sIHRvcFsncHJlY2lvJ10pIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICJsZW4oX2F4KCkucGF0Y2hlcykgPT0gNSBhbmQgc29ydGVkKF93aWR0aHMoKSkgPT0gc29ydGVkKHByb2R1Y3Rvcy5ubGFyZ2VzdCg1LCAncHJlY2lvJylbJ3ByZWNpbyddLnRvbGlzdCgpKSIsICJjdXN0b21fbXNnIjogIkRlYmUgaGFiZXIgNSBiYXJyYXMgaG9yaXpvbnRhbGVzIGNvbiBsb3MgcHJlY2lvcyBkZSBsb3MgNSBwcm9kdWN0b3MgbcOhcyBjYXJvcy4ifSwgImhpbnRzIjogWyJQcmltZXJvIGxvcyBkYXRvczogbG9zIDUgbcOhcyBjYXJvcy4iLCAibmxhcmdlc3QoNSwgJ3ByZWNpbycpIGRhIGVsIHRvcC4iLCAiQmFycmFzIGhvcml6b250YWxlczogcGx0LmJhcmgoY2F0ZWdvcsOtYXMsIHZhbG9yZXMpLiJdLCAiZXhwbGFpbiI6ICJTZSBzZWxlY2Npb25hbiBsb3MgNSBtw6FzIGNhcm9zIGNvbiBgbmxhcmdlc3RgIHkgc2UgZ3JhZmljYW4gY29uIGJhcnJhcyBob3Jpem9udGFsZXMgKG3DoXMgbGVnaWJsZXMgcGFyYSBub21icmVzKS4iLCAicHNldWRvIjogInRvcCDihpAgNSBwcm9kdWN0b3MgbcOhcyBjYXJvc1xuYmFycmFzX2hvcml6b250YWxlcyh0b3BbJ3Byb2R1Y3RvJ10sIHRvcFsncHJlY2lvJ10pIn0sICJwbHQtZDAxIjogeyJzb2x1dGlvbiI6ICJzdG9ja19jYXQgPSBwcm9kdWN0b3MuZ3JvdXBieSgnY2F0ZWdvcmlhJylbJ3N0b2NrJ10uc3VtKClcbnBsdC5iYXIoc3RvY2tfY2F0LmluZGV4LCBzdG9ja19jYXQudmFsdWVzKVxucGx0LnRpdGxlKCdTdG9jayBwb3IgY2F0ZWdvcsOtYScpIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICJsZW4oX2F4KCkucGF0Y2hlcykgPT0gNCBhbmQgc29ydGVkKF9oZWlnaHRzKCkpID09IHNvcnRlZChwcm9kdWN0b3MuZ3JvdXBieSgnY2F0ZWdvcmlhJylbJ3N0b2NrJ10uc3VtKCkudG9saXN0KCkpIiwgImN1c3RvbV9tc2ciOiAiRGViZSBoYWJlciA0IGJhcnJhcyAodW5hIHBvciBjYXRlZ29yw61hKSBjb24gZWwgc3RvY2sgdG90YWwgZGUgY2FkYSB1bmEuIn0sICJoaW50cyI6IFsiUHJpbWVybyBsb3MgZGF0b3M6IGhheSBxdWUgYWdyZWdhciBhbnRlcyBkZSBncmFmaWNhci4iLCAiZ3JvdXBieSgnY2F0ZWdvcmlhJylbJ3N0b2NrJ10uc3VtKCkgdGUgZGEgdW5hIFNlcmllcyBjYXRlZ29yw61hIOKGkiB0b3RhbC4iLCAiRWwgw61uZGljZSB2YSBhIFggeSBsb3MgdmFsb3JlcyBhIFkgZW4gcGx0LmJhci4iXSwgImV4cGxhaW4iOiAiU2UgYWdyZWdhIGNvbiBncm91cGJ5ICh1bmEgZmlsYSBwb3IgY2F0ZWdvcsOtYSkgeSBzZSBncmFmaWNhIGVsIHJlc3VsdGFkbzogbG9zIGdyw6FmaWNvcyBkZSBiYXJyYXMgY2FzaSBzaWVtcHJlIG11ZXN0cmFuIGRhdG9zIHlhIHJlc3VtaWRvcy4iLCAicHNldWRvIjogInN0b2NrX2NhdCDihpAgc3VtYSBkZSBzdG9jayBwb3IgY2F0ZWdvcsOtYVxuYmFycmFzKHN0b2NrX2NhdC5pbmRleCwgc3RvY2tfY2F0LnZhbHVlcykifSwgInBsdC1nMDEiOiB7InNvbHV0aW9uIjogInN0b2NrX2NhdCA9IHByb2R1Y3Rvcy5ncm91cGJ5KCdjYXRlZ29yaWEnKVsnc3RvY2snXS5zdW0oKS5zb3J0X3ZhbHVlcyhhc2NlbmRpbmc9RmFsc2UpXG5wbHQuYmFyKHN0b2NrX2NhdC5pbmRleCwgc3RvY2tfY2F0LnZhbHVlcykiLCAiY2hlY2siOiB7InBsb3QiOiB0cnVlLCAiY3VzdG9tIjogImxlbihfYXgoKS5wYXRjaGVzKSA9PSA0IGFuZCBfaGVpZ2h0cygpID09IHNvcnRlZChwcm9kdWN0b3MuZ3JvdXBieSgnY2F0ZWdvcmlhJylbJ3N0b2NrJ10uc3VtKCkudG9saXN0KCksIHJldmVyc2U9VHJ1ZSkiLCAiY3VzdG9tX21zZyI6ICJEZWJlbiB2ZXJzZSA0IGJhcnJhcyBjb24gZWwgc3RvY2sgcG9yIGNhdGVnb3LDrWEsIGRlIG1heW9yIGEgbWVub3IuIn0sICJoaW50cyI6IFsiwr9Ew7NuZGUgc2UgZGVjaWRlIGVsIG9yZGVuIGRlIGxhcyBiYXJyYXM6IGVuIGVsIGdyw6FmaWNvIG8gZW4gbG9zIGRhdG9zPyIsICJPcmRlbsOhIGxhIFNlcmllcyBhbnRlcyBkZSBncmFmaWNhci4iLCAic29ydF92YWx1ZXMoYXNjZW5kaW5nPUZhbHNlKS4iXSwgImV4cGxhaW4iOiAiRWwgb3JkZW4gZGUgbGFzIGJhcnJhcyBlcyBlbCBvcmRlbiBkZSBsb3MgZGF0b3M6IGJhc3RhIGNvbiBvcmRlbmFyIGxhIFNlcmllcyBhbnRlcyBkZSBncmFmaWNhci4iLCAicHNldWRvIjogInN0b2NrX2NhdCDihpAgc3VtYSBwb3IgY2F0ZWdvcsOtYSwgb3JkZW5hZGEgZGUgbWF5b3IgYSBtZW5vclxuYmFycmFzKHN0b2NrX2NhdC5pbmRleCwgc3RvY2tfY2F0LnZhbHVlcykifSwgInBsdDA3IjogeyJzb2x1dGlvbiI6ICJwbHQuaGlzdCh2ZW50YXNfZGlhcmlhc1sndmVudGFzJ10sIGJpbnM9MTIsIGVkZ2Vjb2xvcj0nYmxhY2snKSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAibGVuKF9heCgpLnBhdGNoZXMpID09IDEyIGFuZCBzdW0oX2hlaWdodHMoKSkgPT0gOTAiLCAiY3VzdG9tX21zZyI6ICJFbCBoaXN0b2dyYW1hIGRlYmUgdGVuZXIgMTIgYmFycmFzIGNvbiBsb3MgOTAgZMOtYXMgZGUgdmVudGFzLiJ9LCAiaGludHMiOiBbIkxhIHByZWd1bnRhIGVzIHNvYnJlIGxhIGRpc3RyaWJ1Y2nDs24gZGUgdW5hIHZhcmlhYmxlLiIsICJIaXN0b2dyYW1hOiBhZ3J1cGEgbG9zIHZhbG9yZXMgZW4gaW50ZXJ2YWxvcy4iLCAicGx0Lmhpc3QodmFsb3JlcywgYmlucz0xMikuIl0sICJleHBsYWluIjogIkVsIGhpc3RvZ3JhbWEgZGl2aWRlIGVsIHJhbmdvIGRlIHZlbnRhcyBlbiAxMiBpbnRlcnZhbG9zIHkgY3VlbnRhIGN1w6FudG9zIGTDrWFzIGNhZSBlbiBjYWRhIHVuby4iLCAicHNldWRvIjogImhpc3RvZ3JhbWEodmVudGFzX2RpYXJpYXNbJ3ZlbnRhcyddLCAxMiBpbnRlcnZhbG9zKSJ9LCAicGx0LWcwMiI6IHsic29sdXRpb24iOiAicGx0Lmhpc3QodmVudGFzX2RpYXJpYXNbJ3Zpc2l0YXMnXSwgYmlucz04KVxucGx0LnRpdGxlKCdEaXN0cmlidWNpw7NuIGRlIHZpc2l0YXMnKSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAibGVuKF9heCgpLnBhdGNoZXMpID09IDggYW5kIHN1bShfaGVpZ2h0cygpKSA9PSA5MCBhbmQgX2F4KCkuZ2V0X3RpdGxlKCkgPT0gJ0Rpc3RyaWJ1Y2nDs24gZGUgdmlzaXRhcyciLCAiY3VzdG9tX21zZyI6ICJFbCBoaXN0b2dyYW1hIGRlYmUgdGVuZXIgOCBiYXJyYXMgY29uIGxvcyA5MCBkw61hcyBkZSB2aXNpdGFzIHkgZWwgdMOtdHVsbyAnRGlzdHJpYnVjacOzbiBkZSB2aXNpdGFzJy4ifSwgImhpbnRzIjogWyLCv1F1w6kgY2FtYmlhIHJlc3BlY3RvIGRlbCBlamVyY2ljaW8gYW50ZXJpb3I/IiwgIk1pc21vIHRpcG8gZGUgZ3LDoWZpY28sIG90cmEgY29sdW1uYSB5IG90cm8gbsO6bWVybyBkZSBiaW5zLiIsICJBZ3JlZ8OhIGVsIHTDrXR1bG8uIl0sICJleHBsYWluIjogIk1pc21vIGdyw6FmaWNvIChoaXN0b2dyYW1hKSwgb3RyYSB2YXJpYWJsZTogbGEgaGFiaWxpZGFkIGVzIHJlY29ub2NlciBsYSBwcmVndW50YSAnZGlzdHJpYnVjacOzbicsIG5vIHJlY29yZGFyIGxhIGzDrW5lYSBkZSBjw7NkaWdvLiIsICJwc2V1ZG8iOiAiaGlzdG9ncmFtYSh2aXNpdGFzLCA4IGludGVydmFsb3MpOyB0w610dWxvIn0sICJwbHQwOCI6IHsic29sdXRpb24iOiAicGx0LnNjYXR0ZXIodmVudGFzX2RpYXJpYXNbJ3Zpc2l0YXMnXSwgdmVudGFzX2RpYXJpYXNbJ3ZlbnRhcyddKVxucGx0LnhsYWJlbCgnVmlzaXRhcycpXG5wbHQueWxhYmVsKCdWZW50YXMnKSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAibGVuKF9heCgpLmNvbGxlY3Rpb25zKSA9PSAxIGFuZCBsZW4oX2F4KCkuY29sbGVjdGlvbnNbMF0uZ2V0X29mZnNldHMoKSkgPT0gOTAgYW5kIF9heCgpLmdldF94bGFiZWwoKSA9PSAnVmlzaXRhcycgYW5kIF9heCgpLmdldF95bGFiZWwoKSA9PSAnVmVudGFzJyIsICJjdXN0b21fbXNnIjogIkRlYmUgaGFiZXIgdW4gc2NhdHRlciBjb24gbG9zIDkwIGTDrWFzIHkgbGFzIGV0aXF1ZXRhcyAnVmlzaXRhcycgeSAnVmVudGFzJy4ifSwgImhpbnRzIjogWyJEb3MgdmFyaWFibGVzIG51bcOpcmljYXMgcG9yIGTDrWE6IMK/cXXDqSBncsOhZmljbz8iLCAiQ2FkYSBwdW50byBlcyB1biBkw61hOiB2aXNpdGFzIGVuIFgsIHZlbnRhcyBlbiBZLiIsICJwbHQuc2NhdHRlcih4LCB5KSArIGV0aXF1ZXRhcyBkZSBlamVzLiJdLCAiZXhwbGFpbiI6ICJDYWRhIHB1bnRvIHJlcHJlc2VudGEgdW4gZMOtYSBjb24gc3VzIHZpc2l0YXMgeSB2ZW50YXM7IGxhIG51YmUgbXVlc3RyYSBzaSBoYXkgcmVsYWNpw7NuLiIsICJwc2V1ZG8iOiAiZGlzcGVyc2nDs24odmlzaXRhcywgdmVudGFzKVxuZWplIFggJ1Zpc2l0YXMnOyBlamUgWSAnVmVudGFzJyJ9LCAicGx0MDkiOiB7InNvbHV0aW9uIjogImNvbnRlbyA9IHByb2R1Y3Rvc1snY2F0ZWdvcmlhJ10udmFsdWVfY291bnRzKClcbnBsdC5waWUoY29udGVvLCBsYWJlbHM9Y29udGVvLmluZGV4LCBhdXRvcGN0PSclMS4wZiUlJykiLCAiY2hlY2siOiB7InBsb3QiOiB0cnVlLCAiY3VzdG9tIjogImxlbihfYXgoKS5wYXRjaGVzKSA9PSA0IGFuZCBsZW4oX2F4KCkudGV4dHMpID09IDgiLCAiY3VzdG9tX21zZyI6ICJMYSB0b3J0YSBkZWJlIHRlbmVyIDQgcG9yY2lvbmVzLCBjb24gc3UgZXRpcXVldGEgeSBzdSBwb3JjZW50YWplIGNhZGEgdW5hICh1c2EgbGFiZWxzIHkgYXV0b3BjdCkuIn0sICJoaW50cyI6IFsiRXMgY29tcG9zaWNpw7NuOiBwYXJ0ZXMgZGUgdW4gdG90YWwgKHBvY2FzIGNhdGVnb3LDrWFzKS4iLCAiUHJpbWVybyBjb250w6EgcHJvZHVjdG9zIHBvciBjYXRlZ29yw61hLiIsICJwbHQucGllKHZhbG9yZXMsIGxhYmVscz0uLi4sIGF1dG9wY3Q9JyUxLjBmJSUnKS4iXSwgImV4cGxhaW4iOiAiTGEgdG9ydGEgbXVlc3RyYSBsYSBwcm9wb3JjacOzbiBkZSBwcm9kdWN0b3MgZGUgY2FkYSBjYXRlZ29yw61hLiBGdW5jaW9uYSBwb3JxdWUgc29uIHNvbG8gNCBwYXJ0ZXM7IGNvbiBtdWNoYXMsIG1lam9yIGJhcnJhcy4iLCAicHNldWRvIjogImNvbnRlbyDihpAgY2FudGlkYWQgZGUgcHJvZHVjdG9zIHBvciBjYXRlZ29yw61hXG50b3J0YShjb250ZW8sIGV0aXF1ZXRhcyA9IGNhdGVnb3LDrWFzLCBwb3JjZW50YWplcykifSwgInBsdDEwIjogeyJzb2x1dGlvbiI6ICJmaWcsIChheDEsIGF4MikgPSBwbHQuc3VicGxvdHMoMSwgMiwgZmlnc2l6ZT0oMTAsIDQpKVxuYXgxLnBsb3QobWVzZXMsIGluZ3Jlc29zKVxuYXgyLmJhcihtZXNlcywgZ2FzdG9zKSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAibGVuKF9maWcoKS5heGVzKSA9PSAyIGFuZCBsZW4oX2F4KDApLmxpbmVzKSA9PSAxIGFuZCBsZW4oX2F4KDEpLnBhdGNoZXMpID09IDYiLCAiY3VzdG9tX21zZyI6ICJJenF1aWVyZGE6IHVuYSBsw61uZWEgY29uIGluZ3Jlc29zLiBEZXJlY2hhOiA2IGJhcnJhcyBjb24gZ2FzdG9zLiJ9LCAiaGludHMiOiBbIlNvbiBkb3MgZ3LDoWZpY29zIGVuIHVuYSBmaWd1cmEsIGxhZG8gYSBsYWRvLiIsICJwbHQuc3VicGxvdHMoMSwgMikgZGV2dWVsdmUgbGEgZmlndXJhIHkgZG9zIGVqZXMuIiwgIkRpYnVqw6EgZW4gY2FkYSBlamUgY29uIGF4LnBsb3QgLyBheC5iYXIuIl0sICJleHBsYWluIjogImBwbHQuc3VicGxvdHMoMSwgMilgIGNyZWEgdW5hIGZpZ3VyYSBjb24gZG9zIGVqZXM7IGNhZGEgZ3LDoWZpY28gc2UgZGlidWphIGNvbiBzdSBvYmpldG8gYGF4YC4iLCAicHNldWRvIjogImZpZywgKGF4MSwgYXgyKSDihpAgc3VicGxvdHMoMSBmaWxhLCAyIGNvbHVtbmFzKVxuYXgxOiBsw61uZWEgZGUgaW5ncmVzb3M7IGF4MjogYmFycmFzIGRlIGdhc3RvcyJ9LCAicGx0LWYwMSI6IHsic29sdXRpb24iOiAiZmlnLCBheHMgPSBwbHQuc3VicGxvdHMoMSwgMiwgZmlnc2l6ZT0oMTEsIDQpKVxuYXhzWzBdLnBsb3QodmVudGFzX2RpYXJpYXNbJ2ZlY2hhJ10sIHZlbnRhc19kaWFyaWFzWyd2ZW50YXMnXSlcbmF4c1swXS5zZXRfdGl0bGUoJ1ZlbnRhcyBkaWFyaWFzJylcbmF4c1sxXS5oaXN0KHZlbnRhc19kaWFyaWFzWyd2ZW50YXMnXSwgYmlucz0xMilcbmF4c1sxXS5zZXRfdGl0bGUoJ0Rpc3RyaWJ1Y2nDs24gZGUgdmVudGFzJylcbmZpZy50aWdodF9sYXlvdXQoKSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAibGVuKF9maWcoKS5heGVzKSA9PSAyIGFuZCBsZW4oX2F4KDApLmxpbmVzKSA9PSAxIGFuZCBsZW4oX2F4KDEpLnBhdGNoZXMpID09IDEyIiwgImN1c3RvbV9tc2ciOiAiSXpxdWllcmRhOiB1bmEgbMOtbmVhOyBkZXJlY2hhOiB1biBoaXN0b2dyYW1hIGRlIDEyIGJhcnJhcy4ifSwgImhpbnRzIjogWyJFamVjdXRhbG8geSByZWxhY2lvbsOhIGNhZGEgbMOtbmVhIGNvbiBsbyBxdWUgc2UgdmUuIiwgIlByb2LDoSBjYW1iaWFyIGF4c1sxXS5zZXRfdGl0bGUgcG9yIHBsdC50aXRsZSB5IG9ic2VydsOhLiIsICJQZW5zw6EgcXXDqSBsZSBmYWx0YSBhIGNhZGEgZWplLiJdLCAiZXhwbGFpbiI6ICJMYSBlc3RydWN0dXJhIGZpZ3VyYS9lamVzIHBlcm1pdGUgY29tcG9uZXIgZ3LDoWZpY29zIHF1ZSByZXNwb25kZW4gcHJlZ3VudGFzIGRpc3RpbnRhcyBzb2JyZSBsb3MgbWlzbW9zIGRhdG9zLiIsICJwc2V1ZG8iOiAiIn0sICJwbHQxMSI6IHsic29sdXRpb24iOiAidmVudGFzX2RpYXJpYXMucGxvdCh4PSdmZWNoYScsIHk9J3ZlbnRhcycsIHRpdGxlPSdWZW50YXMgZGlhcmlhcycpIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICJsZW4oX2F4KCkubGluZXMpID49IDEgYW5kIGxlbihfYXgoKS5saW5lc1swXS5nZXRfeWRhdGEoKSkgPT0gOTAgYW5kIF9heCgpLmdldF90aXRsZSgpID09ICdWZW50YXMgZGlhcmlhcyciLCAiY3VzdG9tX21zZyI6ICJEZWJlIHZlcnNlIHVuYSBsw61uZWEgY29uIGxvcyA5MCBkw61hcyB5IGVsIHTDrXR1bG8gJ1ZlbnRhcyBkaWFyaWFzJy4ifSwgImhpbnRzIjogWyJMb3MgZGF0b3MgeWEgZXN0w6FuIGVuIHVuIERhdGFGcmFtZS4iLCAiTG9zIERhdGFGcmFtZXMgdGllbmVuIHN1IHByb3BpbyBtw6l0b2RvIHBhcmEgZ3JhZmljYXIuIiwgImRmLnBsb3QoeD0uLi4sIHk9Li4uLCB0aXRsZT0uLi4pLiJdLCAiZXhwbGFpbiI6ICJgZGYucGxvdGAgdXNhIG1hdHBsb3RsaWIgcG9yIGRlYmFqbyB5IHBlcm1pdGUgZ3JhZmljYXIgY29sdW1uYXMgZGlyZWN0YW1lbnRlIGluZGljYW5kbyB4LCB5IHkgdMOtdHVsby4iLCAicHNldWRvIjogInZlbnRhc19kaWFyaWFzLnBsb3QoeD0nZmVjaGEnLCB5PSd2ZW50YXMnLCB0aXRsZT0nVmVudGFzIGRpYXJpYXMnKSJ9LCAicGx0LXAyIjogeyJzb2x1dGlvbiI6ICJmaWcsIGF4cyA9IHBsdC5zdWJwbG90cygyLCAyLCBmaWdzaXplPSgxMiwgOCkpXG5heHNbMCwgMF0ucGxvdCh2ZW50YXNfZGlhcmlhc1snZmVjaGEnXSwgdmVudGFzX2RpYXJpYXNbJ3ZlbnRhcyddKVxuYXhzWzAsIDBdLnNldF90aXRsZSgnRXZvbHVjacOzbiBkZSB2ZW50YXMnKVxuYXhzWzAsIDFdLmhpc3QodmVudGFzX2RpYXJpYXNbJ3ZlbnRhcyddLCBiaW5zPTEyKVxuYXhzWzAsIDFdLnNldF90aXRsZSgnRGlzdHJpYnVjacOzbiBkZSB2ZW50YXMnKVxuYXhzWzEsIDBdLnNjYXR0ZXIodmVudGFzX2RpYXJpYXNbJ3Zpc2l0YXMnXSwgdmVudGFzX2RpYXJpYXNbJ3ZlbnRhcyddKVxuYXhzWzEsIDBdLnNldF90aXRsZSgnVmlzaXRhcyB2cyB2ZW50YXMnKVxuc3RvY2tfY2F0ID0gcHJvZHVjdG9zLmdyb3VwYnkoJ2NhdGVnb3JpYScpWydzdG9jayddLnN1bSgpXG5heHNbMSwgMV0uYmFyKHN0b2NrX2NhdC5pbmRleCwgc3RvY2tfY2F0LnZhbHVlcylcbmF4c1sxLCAxXS5zZXRfdGl0bGUoJ1N0b2NrIHBvciBjYXRlZ29yw61hJylcbmZpZy50aWdodF9sYXlvdXQoKSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAibGVuKF9maWcoKS5heGVzKSA9PSA0IGFuZCBsZW4oX2F4KDApLmxpbmVzKSA+PSAxIGFuZCBsZW4oX2F4KDEpLnBhdGNoZXMpID49IDUgYW5kIGxlbihfYXgoMikuY29sbGVjdGlvbnMpID49IDEgYW5kIGxlbihfYXgoMykucGF0Y2hlcykgPT0gNCBhbmQgYWxsKF9heChpKS5nZXRfdGl0bGUoKSBmb3IgaSBpbiByYW5nZSg0KSkiLCAiY3VzdG9tX21zZyI6ICJTZSBlc3BlcmFuIDQgZWplczogbMOtbmVhIChhcnJpYmEgaXpxLiksIGhpc3RvZ3JhbWEgKGFycmliYSBkZXIuKSwgc2NhdHRlciAoYWJham8gaXpxLiksIDQgYmFycmFzIChhYmFqbyBkZXIuKSwgdG9kb3MgY29uIHTDrXR1bG8uIn0sICJoaW50cyI6IFsiU2VndcOtIHR1IGRpc2XDsW8gZWplIHBvciBlamUuIiwgIkNvbiB1bmEgZ3JpbGxhIDLDlzIsIGNhZGEgZWplIHNlIGFjY2VkZSBjb21vIGF4c1tmaWxhLCBjb2x1bW5hXS4iLCAiVXPDoSBheC5wbG90LCBheC5oaXN0LCBheC5zY2F0dGVyLCBheC5iYXIgeSBheC5zZXRfdGl0bGUgZW4gY2FkYSB1bm8uIl0sICJleHBsYWluIjogIkNhZGEgZWplIGRlIGxhIGdyaWxsYSByZXNwb25kZSB1bmEgcHJlZ3VudGEuIFNlIGRpYnVqYSBjb24gZWwgb2JqZXRvIGF4IGNvcnJlc3BvbmRpZW50ZSB5IHNlIHRpdHVsYSBjb24gc2V0X3RpdGxlLiIsICJwc2V1ZG8iOiAiZmlnLCBheHMg4oaQIHN1YnBsb3RzKDIsIDIpXG5wYXJhIGNhZGEgZWplOiBkaWJ1amFyIHN1IGdyw6FmaWNvIHkgcG9uZXJsZSB0w610dWxvXG50aWdodF9sYXlvdXQifSwgInBsdC1wMyI6IHsic29sdXRpb24iOiAiZmlnLCBheHMgPSBwbHQuc3VicGxvdHMoMSwgMiwgZmlnc2l6ZT0oMTAsIDQpKVxuYXhzWzBdLnBsb3QobWVzZXMsIGluZ3Jlc29zKVxuYXhzWzFdLmJhcihtZXNlcywgZ2FzdG9zKSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAibGVuKF9maWcoKS5heGVzKSA9PSAyIGFuZCBsZW4oX2F4KDApLmxpbmVzKSA9PSAxIGFuZCBsZW4oX2F4KDEpLnBhdGNoZXMpID09IDYgYW5kIGxlbihfYXgoMSkubGluZXMpID09IDAiLCAiY3VzdG9tX21zZyI6ICJJenF1aWVyZGE6IGxhIGzDrW5lYSBkZSBpbmdyZXNvcy4gRGVyZWNoYTogc29sbyBsYXMgNiBiYXJyYXMgZGUgZ2FzdG9zLiJ9LCAiaGludHMiOiBbIk1pcsOhIGVsIGdyw6FmaWNvOiDCv3F1w6kgZWplIHF1ZWTDsyB2YWPDrW8/IiwgIsK/RW4gcXXDqSBlamUgZGlidWphIHBsdC5wbG90IGN1YW5kbyBoYXkgdmFyaW9zPyIsICJVc8OhIGF4c1swXSB5IGF4c1sxXS4iXSwgImV4cGxhaW4iOiAiRGVzcHXDqXMgZGUgYHBsdC5zdWJwbG90c2AsIGVsIGVqZSAnYWN0aXZvJyBlcyBlbCDDumx0aW1vIGNyZWFkby4gYHBsdC5wbG90YCB5IGBwbHQuYmFyYCBkaWJ1amFuIGFow60sIGFzw60gcXVlIGFtYm9zIGdyw6FmaWNvcyBxdWVkYW4gZW4gZWwgZWplIGRlcmVjaG8uIEhheSBxdWUgZGlidWphciBleHBsw61jaXRhbWVudGUgY29uIGBheHNbMF1gIHkgYGF4c1sxXWAuIiwgInBzZXVkbyI6ICJheHNbMF06IGzDrW5lYSBkZSBpbmdyZXNvc1xuYXhzWzFdOiBiYXJyYXMgZGUgZ2FzdG9zIn19'
_TABLAS = ['meses', 'ingresos', 'gastos', 'productos', 'ventas_diarias']

import base64 as _b64, json as _json, re as _re, warnings as _w
_w.simplefilter("ignore")
_SOL = _json.loads(_b64.b64decode(_SOL_B64).decode("utf-8"))

def reiniciar_datos():
    """Vuelve a cargar todos los datos limpios (cada ejercicio empieza de cero)."""
    globals().pop("resultado", None)
    exec(_SETUP, globals())
    pq_prepare()

def _codigo_celda():
    try:
        src = get_ipython().history_manager.input_hist_raw[-1]
    except Exception:
        return ""
    src = "\n".join(l for l in src.splitlines() if "comprobar(" not in l)
    return "\n".join(_re.sub(r"#.*$", "", l) for l in src.splitlines())

def comprobar(ej):
    info = _SOL[ej]
    def esperado():
        exp_ns = {}
        exec(_SETUP, exp_ns)
        exec(info["solution"], exp_ns)
        return exp_ns
    g = globals()
    g["_pq_code"] = _codigo_celda()
    ok, msg = pq_check(g, esperado, info["check"])
    from IPython.display import HTML, display
    import html as _h
    if ok:
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#0f9d6e;color:#fff;'
                     'font:600 15px system-ui">✅ Verificado. Ahora explicá con tus palabras por qué funciona '
                     'y probá una variante.</div>'))
    else:
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#b4235a;color:#fff;'
                     'font:500 15px system-ui">Todavía no. ' + _h.escape(msg) + '</div>'))
    if "resultado" in g:
        return g["resultado"]

def ver_pista(ej, nivel=1):
    """Pistas progresivas (1-3). Pedí la siguiente solo si la anterior no alcanzó."""
    pistas = _SOL[ej]["hints"]
    print(f"Pista {nivel}/{len(pistas)}: {pistas[min(nivel, len(pistas)) - 1]}")

def ver_solucion(ej):
    """Último recurso: primero la idea, después el pseudocódigo y al final el código."""
    info = _SOL[ej]
    print("💡 Idea:\n" + info["explain"] + "\n")
    print("🧭 Pseudocódigo:\n" + info["pseudo"] + "\n")
    print("🧑‍💻 Código:\n" + info["solution"])

reiniciar_datos()
print("✅ Datos cargados:", ", ".join(_TABLAS))


---
# 📌 Evolución en el tiempo: líneas

## 1. Tu primer gráfico
`Código` · `Muy fácil`

Mostrá cómo evolucionaron los `ingresos` a lo largo de los `meses`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

# plt ya está importado


comprobar("plt01")

## 2. Título y etiquetas
`Código` · `Fácil`

Graficá los `ingresos` por mes y hacé que se entienda solo: título **"Ingresos por mes"**, eje X **"Mes"** y eje Y **"Ingresos (miles)"** (exactamente así).

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

plt.plot(meses, ingresos)
# añade título y etiquetas


comprobar("plt02")

## 3. Comparar con leyenda
`Código` · `Medio`

Implementá tu plan: `ingresos` y `gastos` por mes en el mismo gráfico, con las etiquetas **"Ingresos"** y **"Gastos"** y la leyenda visible.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 



comprobar("plt03")

## 4. La leyenda vacía
`Debugging` · `Fácil`

Este gráfico debería mostrar una leyenda con Ingresos y Gastos, pero la leyenda no aparece. Corregilo.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

plt.plot(meses, ingresos)
plt.plot(meses, gastos)
plt.legend()

comprobar("plt-e01")

## 5. Los ejes al revés
`Debugging` · `Fácil`

Este gráfico de evolución quedó con los meses en el eje vertical. Corregilo para que el tiempo vaya en X.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

plt.plot(ingresos, meses)

comprobar("plt-e02")

## 6. Dale estilo
`Código` · `Medio`

Graficá `ingresos` por mes en color **"green"**, con marcador **"o"** y línea de guiones **"--"**.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

plt.plot(meses, ingresos)

comprobar("plt04")

---
# 📌 Comparar categorías: barras

## 7. Gráfico de barras
`Código` · `Muy fácil`

¿Qué productos tienen más (y menos) stock? Dibujá un gráfico con el `stock` de cada `producto` de la tabla `productos`, en el orden de la tabla.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 



comprobar("plt05")

## 8. Barras horizontales: un top 5
`Código` · `Medio`

Mostrá un **top 5 de productos más caros** con barras horizontales (`producto` y `precio`).

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

top = 


comprobar("plt06")

## 9. Stock total por categoría
`Código` · `Medio`

¿Qué categoría tiene más unidades en stock? Calculá el **stock total por categoría** y graficalo con barras (una barra por categoría).

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 



comprobar("plt-d01")

## 10. Transferencia: ordenar las barras
`Transferencia` · `Medio`

Graficá el **stock total por categoría** con las barras ordenadas de **mayor a menor**.

> 🔁 **Qué cambia:** Antes graficaste el stock por categoría en orden alfabético. Ahora querés que las barras queden **ordenadas de mayor a menor** para facilitar la comparación.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 



comprobar("plt-g01")

---
# 📌 Distribuciones: histogramas

## 11. Histograma
`Código` · `Medio`

¿Cómo se distribuyen las ventas diarias? Dibujá la distribución de `ventas` de `ventas_diarias` con **12 intervalos (bins)**.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 



comprobar("plt07")

## 12. Transferencia: distribución de visitas
`Transferencia` · `Fácil`

Dibujá la distribución de `visitas` de `ventas_diarias` con **8 intervalos** y el título **"Distribución de visitas"**.

> 🔁 **Qué cambia:** Misma pregunta (distribución), otra variable y otra cantidad de intervalos.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 



comprobar("plt-g02")

---
# 📌 Relaciones: dispersión

## 13. Dispersión: ¿hay relación?
`Código` · `Medio`

¿Los días con más visitas tienen más ventas? Dibujá cada día como un punto (`visitas` en X, `ventas` en Y) con las etiquetas de eje **"Visitas"** y **"Ventas"**.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 



comprobar("plt08")

## 14. Gráfico de torta
`Código` · `Medio`

¿Qué proporción del catálogo es de cada `categoria`? Contá los productos por categoría y dibujá una torta con la etiqueta de cada categoría y su porcentaje (`autopct`).

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

conteo = productos['categoria'].value_counts()


comprobar("plt09")

---
# 📌 Varios gráficos y pandas

## 15. Varios gráficos juntos
`Código` · `Difícil`

Armá una figura con **2 gráficos lado a lado**: a la izquierda, la evolución de `ingresos` por mes (línea); a la derecha, los `gastos` por mes (barras).

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))


comprobar("plt10")

## 16. La IA arma un mini dashboard
`IA copiloto` · `Medio`

Pediste a una IA 'un dashboard de 2 gráficos con ventas diarias'. Te dio esto. Ejecutalo y respondé.

> 🤖 Prompt usado:
> Necesito una figura con dos gráficos lado a lado a partir de ventas_diarias: la evolución de las ventas y su distribución. Explicame la estructura figura/ejes.

Antes de ejecutarlo, explicá en comentarios qué hace cada línea.

In [ ]:
reiniciar_datos()  # datos limpios

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].plot(ventas_diarias['fecha'], ventas_diarias['ventas'])
axs[0].set_title('Ventas diarias')
axs[1].hist(ventas_diarias['ventas'], bins=12)
axs[1].set_title('Distribución de ventas')
fig.tight_layout()

comprobar("plt-f01")

## 17. Graficar directo desde pandas
`Código` · `Medio`

Sin usar `plt.plot`, graficá directamente desde el DataFrame `ventas_diarias` las `ventas` a lo largo de la `fecha`, con el título **"Ventas diarias"**.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 



comprobar("plt11")

---
# 📌 Proyecto: dashboard básico de ventas

## 18. Fase 2 · Construir el dashboard
`Código` · `Difícil`

Construí el dashboard 2×2 diseñado: arriba a la izquierda la **línea** de ventas por fecha, arriba a la derecha el **histograma** de ventas, abajo a la izquierda el **scatter** visitas vs ventas y abajo a la derecha las **barras** de stock por categoría. **Cada gráfico con su título.**

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

fig, axs = plt.subplots(2, 2, figsize=(12, 8))


comprobar("plt-p2")

## 19. Fase 3 · Todo en el mismo gráfico
`Debugging` · `Medio`

Un compañero armó una versión de 2 gráficos, pero los dos quedaron dibujados en el de la derecha y el de la izquierda está vacío. Corregilo.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
plt.plot(meses, ingresos)
plt.bar(meses, gastos)

comprobar("plt-p3")

---
# 🏁 Fin del notebook
Volvé a la [plataforma](https://rosalesluciano.github.io/datalab/) para los ejercicios de reconocimiento, razonamiento y el modo **¿Sé esto?**.